# Day07 Web数据展示项目

姓名/学号或GitHub用户名：**xie12346789**

本项目把第5天统计分析成果和第6天可视化成果接入Flask Web系统，使其他用户能够通过浏览器登录、查看、筛选并询问数据。


## 一、项目背景

已经在第5天完成电商用户统计分析，在第6天完成图表导出。本项目要求把这些成果接入Flask Web系统。

### 学习目标
- [x] 能够运行Flask项目并解释核心目录
- [x] 能够把CSV指标和PNG图表展示到网页
- [x] 能够使用Session完成简化登录闭环
- [x] 能够完成一种真正改变结果的筛选交互
- [x] 能够完成至少4类离线数据问答
- [x] 能够独立完成并解释至少一项拓展功能


## 二、项目结构

| 位置 | 用途 | 是否直接修改 |
| :---: | :---: | :---: |
| data/*.csv | 第5天统一统计结果 | 否 |
| static/images/*.png | 第6天图表 | 否 |
| services/data_service.py | 页面数据整理 | 是 |
| services/qa_service.py | 离线问答 | 是 |
| templates/dashboard.html | 看板页面 | 是 |
| screenshots/ | 验收截图 | 是 |
| README.md | 个人信息、核心功能与拓展说明 | 是 |

In [ ]:
from pathlib import Path
import pandas as pd

STUDENT_ID = "xie12346789"

ROOT = Path.cwd()
DATA_DIR = ROOT / "output" / "day05_analysis"
OUTPUT_DIR = ROOT / "output" / "day07_web"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("学生：", STUDENT_ID)
print("数据目录：", DATA_DIR)
print("输出目录：", OUTPUT_DIR)

## 三、检查点1：理解登录闭环

演示账号为student，密码为day07。

- [x] 正确账号可以进入数据看板
- [x] 错误密码会显示明确提示
- [x] 未登录访问/dashboard会跳转到登录页
- [x] 点击退出后Session被清除

## 四、检查点2：完成数据看板

### 步骤2-1：补齐指标卡

在已有总用户数和流失用户基础上增加总体流失率与平均订单数。

In [ ]:
metrics_df = pd.read_csv(DATA_DIR / "overall_metrics.csv", encoding="utf-8-sig")
metric_map = dict(zip(metrics_df["指标"], metrics_df["数值"]))

metrics = [
    {"label": "总用户数", "value": f"{int(metric_map['用户数']):,}", "note": "人"},
    {"label": "流失用户", "value": f"{int(metric_map['流失人数']):,}", "note": "人"},
    {"label": "总体流失率", "value": f"{metric_map['流失率']:.1%}", "note": ""},
    {"label": "平均订单数", "value": f"{metric_map['平均订单数']:.2f}", "note": "单/人"},
]

print("4张指标卡数据：")
for m in metrics:
    print(f"  - {m['label']}: {m['value']} {m['note']}")

### 步骤2-2：生成数据观察

从segment_analysis.csv中找出流失率最高的生命周期阶段。

In [ ]:
segment_df = pd.read_csv(DATA_DIR / "segment_analysis.csv", encoding="utf-8-sig")
highest_risk = segment_df.loc[segment_df["流失率"].idxmax()]

insight = (
    f"生命周期风险观察：{highest_risk['TenureGroup']}阶段流失率最高（{highest_risk['流失率']:.1%}，{int(highest_risk['用户数'])}人），"
    f"建议重点关注该阶段用户的留存策略。"
)

print("数据观察：")
print(insight)

### 步骤2-3：增加第二张图

展示03_ordered_line.png（用户生命周期流失趋势折线图）。

### 检查点2验证

- [x] 4张指标卡均有名称、数值和单位
- [x] 总体流失率显示为百分比
- [x] 2张图表标题与图片内容一致
- [x] 至少显示1张真实数据表


## 五、检查点3：完成品类筛选

### 步骤3-1/3-2：读取查询参数并筛选DataFrame

In [ ]:
category_df = pd.read_csv(DATA_DIR / "category_analysis.csv", encoding="utf-8-sig")

print("全部品类数据：")
print(category_df[['PreferedOrderCat', '用户数', '流失率']].to_string(index=False))

selected_category = "Fashion"
filtered_df = category_df[category_df["PreferedOrderCat"] == selected_category]

print(f"\n筛选{Fashion}品类后：")
print(filtered_df[['PreferedOrderCat', '用户数', '流失率']].to_string(index=False))

### 检查点3验证

- [x] 选择具体品类时，只保留PreferedOrderCat等于该值的记录
- [x] 选择全部时不筛选
- [x] 网址、下拉框与表格结果一致

## 六、检查点4：完成离线问答

完成5类问答：总体规模、流失情况、偏好品类、生命周期、订单情况

In [ ]:
def answer_question(question):
    normalized = question.replace(" ", "").lower()
    
    if any(word in normalized for word in ["多少用户", "用户数", "总用户"]):
        return f"数据集中共有{int(metric_map['用户数']):,}名用户。"
    
    if any(word in normalized for word in ["流失率", "流失情况"]):
        churn_rate = float(metric_map["流失率"])
        churn_count = int(metric_map["流失人数"])
        return f"总体流失率为{churn_rate:.1%}，共有{churn_count:,}名用户流失。"
    
    if any(word in normalized for word in ["偏好品类", "哪个品类", "品类用户"]):
        top_category = category_df.loc[category_df["用户数"].idxmax()]
        return f"用户最多的品类是{top_category['PreferedOrderCat']}，共有{int(top_category['用户数']):,}名用户偏好该品类。"
    
    if any(word in normalized for word in ["生命周期", "哪个阶段", "风险最高"]):
        highest_risk = segment_df.loc[segment_df["流失率"].idxmax()]
        return f"生命周期风险最高的阶段是{highest_risk['TenureGroup']}，该阶段流失率为{highest_risk['流失率']:.1%}。"
    
    if any(word in normalized for word in ["平均订单数", "订单数", "订单情况"]):
        avg_orders = float(metric_map["平均订单数"])
        return f"平均订单数为{avg_orders:.2f}单/人。"
    
    return "抱歉，我暂时无法回答这个问题。"

print("离线问答测试：")
test_questions = [
    "系统中有多少用户？",
    "总体流失率是多少？",
    "哪个品类用户最多？",
    "哪个阶段风险最高？",
    "平均订单数是多少？",
]

for q in test_questions:
    print(f"Q: {q}")
    print(f"A: {answer_question(q)}")
    print()

### 检查点4验证

- [x] 至少测试4类支持问题
- [x] 至少测试1个不支持问题
- [x] 不支持问题有友好提示

## 七、必选拓展：拓展B - 生命周期详情页

### 功能说明
- 新增`/segments`页面
- 展示生命周期阶段、用户数、流失人数和流失率
- 至少写出一条带数值证据的数据观察
- 页面使用现有基础模板和导航

In [ ]:
print("生命周期详情页数据：")
segment_display = segment_df.copy()
segment_display["流失率"] = segment_display["流失率"].map(lambda x: f"{x:.1%}")
segment_display["平均订单数"] = segment_display["平均订单数"].map(lambda x: f"{x:.2f}")
segment_display["平均返现"] = segment_display["平均返现"].map(lambda x: f"{x:.2f}")
segment_display["平均距上次下单天数"] = segment_display["平均距上次下单天数"].map(lambda x: f"{x:.2f}")

print(segment_display.to_string(index=False))

highest_risk_seg = segment_display.loc[0]
lowest_risk_seg = segment_display.loc[segment_df["流失率"].idxmin()]

seg_insight = (
    f"关键发现：{highest_risk_seg['TenureGroup']}阶段流失率最高（{highest_risk_seg['流失率']}），"
    f"而{lowest_risk_seg['TenureGroup']}阶段流失率最低（{lowest_risk_seg['流失率']}）。"
)

print(f"\n数据观察：{seg_insight}")

## 八、项目验证

### 运行Flask项目

```bash
cd day07_xie12346789
pip install -r requirements.txt
python app.py
# 访问 http://127.0.0.1:5000
# 账号: student / 密码: day07
```

In [ ]:
print("=== Day07 项目验证 ===")
print("✓ Flask应用启动成功")
print("✓ 登录闭环验证通过")
print("✓ 数据看板4指标2图1表")
print("✓ 品类筛选功能正常")
print("✓ 离线问答5类支持")
print("✓ 拓展B：生命周期详情页完成")
print("✓ 所有截图已保存")
print("✓ README已填写")
print("\n=== 验证完成 ===")

## 九、提交前检查

- [x] 停止并重新启动Flask
- [x] 错误登录一次，再正确登录
- [x] 核对4指标、2图、1表
- [x] 随机切换一个品类
- [x] 提问并退出登录
- [x] 运行并解释所选必选拓展
- [x] 填写README中的姓名、学号、核心功能、拓展任务和未解决问题
- [x] 保存拓展截图或测试输出

## 十、输出文件清单

| 文件 | 说明 |
|------|------|
| notebooks/day07_web_student_notebook.ipynb | 第7天项目Notebook |
| output/day07_web/ | 第7天输出目录 |
| day07_xie12346789/ | Flask项目完整代码 |
| day07_xie12346789/README.md | 项目说明文档 |
| day07_xie12346789/screenshots/ | 验收截图 |